In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()  # FUNCTION CALL
API_KEY = os.getenv("OPENAQ_API_KEY")  # METHOD CALL / NAMED INSTANCE 

In [13]:
response = requests.get(  # MAJOR METHOD CALL 
    "https://api.openaq.org/v3/locations",
    params={"coordinates": "41.3874,2.1686",
    "radius": 4500, # Limiting to 2.5 KMs to keep the AQMS in city center
    "limit": 100},
    headers={"X-API-Key": API_KEY},
)
print(f"Response body: {response.text}")
response.raise_for_status()  # METHOD CALL



Response body: {"meta":{"name":"openaq-api","website":"/","page":1,"limit":100,"found":13},"results":[{"id":3273,"name":"BARCELONA (CIUTADELLA)","locality":"BARCELONA","timezone":"Europe/Madrid","country":{"id":67,"code":"ES","name":"Spain"},"owner":{"id":4,"name":"Unknown Governmental Organization"},"provider":{"id":70,"name":"EEA"},"isMobile":false,"isMonitor":true,"instruments":[{"id":2,"name":"Government Monitor"}],"sensors":[{"id":4274503,"name":"no µg/m³","parameter":{"id":19843,"name":"no","units":"µg/m³","displayName":"NO mass"}},{"id":7120,"name":"no2 µg/m³","parameter":{"id":5,"name":"no2","units":"µg/m³","displayName":"NO₂ mass"}},{"id":4275860,"name":"nox µg/m³","parameter":{"id":27,"name":"nox","units":"µg/m³","displayName":"NOx mass"}},{"id":7125,"name":"o3 µg/m³","parameter":{"id":3,"name":"o3","units":"µg/m³","displayName":"O₃ mass"}},{"id":4285579,"name":"pm10 µg/m³","parameter":{"id":1,"name":"pm10","units":"µg/m³","displayName":"PM10"}}],"coordinates":{"latitude":41.

In [ ]:

results = response.json().get("results", [])  # METHOD CALL chained with METHOD CALL
locations = pd.DataFrame([
    {
        "location": r.get("name"),  # METHOD CALL
        "parameters": [s["parameter"]["name"] for s in r.get("sensors", [])],  # METHOD CALL
        "count": len(r.get("sensors", [])),  # FUNCTION & METHOD CALL
        "id": r.get("id"),  # METHOD CALL
        "provider": r.get("provider", {}).get("name"),
        "latitude": r.get("coordinates", {}).get("latitude"),
        "longitude": r.get("coordinates", {}).get("longitude")
    }
    for r in results
])
print(locations)  

results = response.json().get("results", [])  # METHOD CALL chained with METHOD CALL
print(results)  # CHECKING WHAT's INSIDE THE FILE

                             location  \
0              BARCELONA (CIUTADELLA)   
1                           Barcelona   
2                           Barcelona   
3                   BARCELONA (SANTS)   
4             BARCELONA (EL POBLENOU)   
5              BARCELONA (L'EIXAMPLE)   
6                          La Sagrera   
7   BARCELONA (GRÀCIA - SANT GERVASI)   
8             BARCELONA (PALAU REIAL)   
9                          Can Bruixa   
10                Estación de Franca    
11                          Barcelona   
12      Barcelona - Camp d'en Grassot   

                                           parameters  count       id  \
0                            [no, no2, nox, o3, pm10]      5     3273   
1                   [co, no, no2, nox, o3, pm10, so2]      7     3276   
2             [co, no, no2, nox, o3, pm10, pm25, so2]      8     3291   
3                                      [no, no2, nox]      3     3293   
4                                [no, no2, nox, pm10]      4

In [19]:
# keep only official stations with pm25
official_providers = ['Catalonia Medi Ambient I Sostenibilitat', 'EEA']

filtered_stations = locations[
    locations['parameters'].apply(lambda x: 'pm25' in x) &
    locations['provider'].isin(official_providers)
]

print(filtered_stations[['location', 'provider', 'id']])

                  location                                 provider       id
2                Barcelona  Catalonia Medi Ambient I Sostenibilitat     3291
5   BARCELONA (L'EIXAMPLE)                                      EEA     3298
11               Barcelona  Catalonia Medi Ambient I Sostenibilitat  6187808


In [20]:
print(filtered_stations[['location', 'provider', 'id', 'latitude', 'longitude']])

                  location                                 provider       id  \
2                Barcelona  Catalonia Medi Ambient I Sostenibilitat     3291   
5   BARCELONA (L'EIXAMPLE)                                      EEA     3298   
11               Barcelona  Catalonia Medi Ambient I Sostenibilitat  6187808   

    latitude  longitude  
2   41.38749   2.115200  
5   41.38530   2.153800  
11  41.41591   2.187094  


In [21]:
# drop station with no 2022 data
filtered_stations = filtered_stations[filtered_stations['id'] != 6187808]
print(filtered_stations[['location', 'provider', 'id', 'latitude', 'longitude']])

                 location                                 provider    id  \
2               Barcelona  Catalonia Medi Ambient I Sostenibilitat  3291   
5  BARCELONA (L'EIXAMPLE)                                      EEA  3298   

   latitude  longitude  
2  41.38749     2.1152  
5  41.38530     2.1538  


In [23]:
#The PM2.5 sensor ID for 3291 is 4273113
# For 3298 PM2.5 sensor ID is 5079791

In [26]:
#For 3291, let's put 4273113
response = requests.get(
    f"https://api.openaq.org/v3/sensors/4273113/measurements",
    params= {
    "date_from": "2022-01-01",     # Start date
    "date_to": "2022-12-31",       # End date
    "limit": 1000                  # Max results per request
    },
    headers={"X-API-Key": API_KEY},
)
response.raise_for_status()
measurements_data = response.json().get("results", [])

# Convert to DataFrame
df = pd.DataFrame([
    {
        "datetime": m["period"]["datetimeFrom"]["local"],
        "pm25": m["value"]
    }
    for m in measurements_data
])


In [27]:
df

,datetime,pm25
0,2023-03-30T03:00:00+02:00,4.0
1,2023-03-30T11:00:00+02:00,11.0
2,2023-04-01T03:00:00+02:00,16.0
3,2023-04-02T03:00:00+02:00,2.0
4,2023-04-03T02:00:00+02:00,1.0
...,...,...
976,2026-05-24T03:00:00+02:00,8.0
977,2026-05-25T03:00:00+02:00,8.0
978,2026-05-26T03:00:00+02:00,8.0
979,2026-05-27T03:00:00+02:00,8.0


In [28]:
# No data for 2022. Lets try the other. 

In [29]:
#For the other, let's put 5079791
response = requests.get(
    f"https://api.openaq.org/v3/sensors/5079791/measurements",
    params= {
    "date_from": "2022-01-01",     # Start date
    "date_to": "2022-12-31",       # End date
    "limit": 1000                  # Max results per request
    },
    headers={"X-API-Key": API_KEY},
)
response.raise_for_status()
measurements_data = response.json().get("results", [])

# Convert to DataFrame
df = pd.DataFrame([
    {
        "datetime": m["period"]["datetimeFrom"]["local"],
        "pm25": m["value"]
    }
    for m in measurements_data
])


In [30]:
df

,datetime,pm25
0,2023-05-07T03:00:00+02:00,9.0
1,2023-05-09T03:00:00+02:00,13.0
2,2023-05-10T03:00:00+02:00,5.0
3,2023-05-11T03:00:00+02:00,7.0
4,2023-05-12T03:00:00+02:00,2.0
...,...,...
995,2023-11-14T14:00:00+01:00,6.0
996,2023-11-14T15:00:00+01:00,9.0
997,2023-11-14T16:00:00+01:00,11.0
998,2023-11-14T17:00:00+01:00,12.0


In [32]:
# search wider area - all of Catalonia, not just Barcelona center
response = requests.get(
    "https://api.openaq.org/v3/locations",
    params={
        "coordinates": "41.3874,2.1686",
        "radius": 10000,        # 50km radius - all of Catalonia
        "limit": 100,
        "parameters_id": 2,     # 2 is the id for pm25
    },
    headers={"X-API-Key": API_KEY},
)
response.raise_for_status()
results = response.json().get("results", [])

# build DataFrame with date information
stations_2022 = pd.DataFrame([
    {
        "location": r.get("name"),
        "provider": r.get("provider", {}).get("name"),
        "id": r.get("id"),
        "latitude": r.get("coordinates", {}).get("latitude"),
        "longitude": r.get("coordinates", {}).get("longitude"),
        "date_first": r.get("datetimeFirst", {}).get("utc"),
        "date_last": r.get("datetimeLast", {}).get("utc"),
    }
    for r in results
])

# filter for stations that have 2022 data
stations_2022 = stations_2022[
    stations_2022['date_first'] <= '2022-12-31'
]

print(stations_2022[['location', 'provider', 'date_first', 'date_last', 'id']])
print(f"\nTotal stations with 2022 pm25 data: {len(stations_2022)}")

                               location  \
0  BARCELONA (PARC DE LA VALL D'HEBRON)   
1                   Sant Adrià de Besòs   
2                             Barcelona   
4           Hospitalet de Llobregat, l'   

                                  provider            date_first  \
0                                      EEA  2016-11-17T23:00:00Z   
1  Catalonia Medi Ambient I Sostenibilitat  2016-11-17T23:00:00Z   
2  Catalonia Medi Ambient I Sostenibilitat  2016-11-17T23:00:00Z   
4  Catalonia Medi Ambient I Sostenibilitat  2016-11-17T23:00:00Z   

              date_last    id  
0  2026-05-28T09:00:00Z  2990  
1  2026-05-28T09:00:00Z  3229  
2  2026-05-28T02:00:00Z  3291  
4  2026-05-28T09:00:00Z  3316  

Total stations with 2022 pm25 data: 4


In [33]:
print(stations_2022[['location', 'provider', 'id', 'latitude', 'longitude']])

                               location  \
0  BARCELONA (PARC DE LA VALL D'HEBRON)   
1                   Sant Adrià de Besòs   
2                             Barcelona   
4           Hospitalet de Llobregat, l'   

                                  provider    id   latitude  longitude  
0                                      EEA  2990  41.426100   2.148000  
1  Catalonia Medi Ambient I Sostenibilitat  3229  41.425594   2.222200  
2  Catalonia Medi Ambient I Sostenibilitat  3291  41.387490   2.115200  
4  Catalonia Medi Ambient I Sostenibilitat  3316  41.370476   2.114999  


In [34]:
# get pm25 sensor id for each station
for r in results:
    if r.get("id") in stations_2022['id'].values:
        pm25_sensors = [s for s in r.get("sensors", []) if s["parameter"]["name"] == "pm25"]
        if pm25_sensors:
            print(f"Station {r['id']} ({r['name']}): sensor id = {pm25_sensors[0]['id']}")

Station 2990 (BARCELONA (PARC DE LA VALL D'HEBRON)): sensor id = 14095263
Station 3229 (Sant Adrià de Besòs): sensor id = 15147153
Station 3291 (Barcelona): sensor id = 4273113
Station 3316 (Hospitalet de Llobregat, l'): sensor id = 15147149


In [ ]:
import time

# define our stations
stations_info = [
    {"id": 2990, "name": "Vall d'Hebron", "sensor_id": 14095263},
    {"id": 3229, "name": "Sant Adrià de Besòs", "sensor_id": 15147153},
    {"id": 3291, "name": "Barcelona", "sensor_id": 4273113},
    {"id": 3316, "name": "Hospitalet de Llobregat", "sensor_id": 15147149},
]

all_measurements = []

for station in stations_info:
    response = requests.get(
        f"https://api.openaq.org/v3/sensors/{station['sensor_id']}/measurements/daily",
        params={
            "date_from": "2022-01-01",
            "date_to": "2022-12-31",
            "limit": 500,
        },
        headers={"X-API-Key": API_KEY},
    )
    response.raise_for_status()
    results_m = response.json().get("results", [])
    
    for m in results_m:
        all_measurements.append({
            "station_id": station["id"],
            "station_name": station["name"],
            "date": m["period"]["dateFrom"]["local"],
            "pm25": m["value"]
        })
    
    print(f"Got {len(results_m)} records for {station['name']}")
    time.sleep(1)  # be polite to the API

df_measurements = pd.DataFrame(all_measurements)
print(df_measurements)

Got 135 records for Vall d'Hebron
Got 130 records for Sant Adrià de Besòs
Got 500 records for Barcelona
Got 130 records for Hospitalet de Llobregat
     station_id             station_name                       date   pm25
0          2990            Vall d'Hebron  2025-09-10T00:00:00+02:00   4.00
1          2990            Vall d'Hebron  2025-09-11T00:00:00+02:00   5.75
2          2990            Vall d'Hebron  2025-09-12T00:00:00+02:00   6.78
3          2990            Vall d'Hebron  2025-09-13T00:00:00+02:00   7.88
4          2990            Vall d'Hebron  2025-09-14T00:00:00+02:00   6.21
..          ...                      ...                        ...    ...
890        3316  Hospitalet de Llobregat  2026-05-24T00:00:00+02:00  10.00
891        3316  Hospitalet de Llobregat  2026-05-25T00:00:00+02:00  14.00
892        3316  Hospitalet de Llobregat  2026-05-26T00:00:00+02:00  16.00
893        3316  Hospitalet de Llobregat  2026-05-27T00:00:00+02:00  13.00
894        3316  Hospitalet

In [38]:
response = requests.get(
    f"https://api.openaq.org/v3/sensors/4273113/hours",
    params={
        "date_from": "2022-01-01",
        "date_to": "2022-03-31",  # just one quarter first to test
        "limit": 500,
    },
    headers={"X-API-Key": API_KEY},
)
response.raise_for_status()
results_test = response.json().get("results", [])
print(f"Records found: {len(results_test)}")
if results_test:
    print(results_test[0])

Records found: 500
{'value': 4.0, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None}, 'period': {'label': '1hour', 'interval': '01:00:00', 'datetimeFrom': {'utc': '2023-03-30T01:00:00Z', 'local': '2023-03-30T03:00:00+02:00'}, 'datetimeTo': {'utc': '2023-03-30T02:00:00Z', 'local': '2023-03-30T04:00:00+02:00'}}, 'coordinates': None, 'summary': {'min': 4.0, 'q02': 4.0, 'q25': 4.0, 'median': 4.0, 'q75': 4.0, 'q98': 4.0, 'max': 4.0, 'avg': 4.0, 'sd': None}, 'coverage': {'expectedCount': 1, 'expectedInterval': '01:00:00', 'observedCount': 1, 'observedInterval': '01:00:00', 'percentComplete': 100.0, 'percentCoverage': 100.0, 'datetimeFrom': {'utc': '2023-03-30T01:00:00Z', 'local': '2023-03-30T03:00:00+02:00'}, 'datetimeTo': {'utc': '2023-03-30T02:00:00Z', 'local': '2023-03-30T04:00:00+02:00'}}}


In [ ]:
# NO data found I guess. 

In [ ]:
# # Create folder if it doesn't exist
# os.makedirs("data/raw", exist_ok=True)

# # Save to CSV
# df.to_csv("data/raw/barcelona_pm25.csv", index=False)
# print(f"Saved {len(df)} records to data/raw/barcelona_pm25.csv")